# Hermes-Astra × JEPA — carnet d'entraînement

Ce carnet entraîne un **JEPA** (*Joint-Embedding Predictive Architecture*, LeCun 2022 ; I-JEPA, Assran et al. 2023) sur des bougies de 5 minutes de contrats perpétuels crypto, puis mesure honnêtement si la représentation apprise contient de quoi trader **aux règles exactes d'Hermes** (levier 15, frais taker 0,05 % par jambe, TP / SL / trail en pourcentage de marge, durée maximale, une position par instrument).

**Ce qu'il fait, dans l'ordre**

1. Réglages (mode rapide ou complet).
2. Données : téléchargement des archives mensuelles publiques de Binance Futures (`data.binance.vision`, sans compte ni clé). Repli automatique sur l'API publique OKX si le seau Binance est inaccessible.
3. Caractéristiques causales par bougie, normalisation robuste calculée sur l'entraînement seul.
4. Modèle : encodeur de contexte, encodeur cible (moyenne mobile exponentielle), prédicteur. Le contexte est 24 h de bougies, la cible est les 4 h qui suivent. La perte est mesurée dans l'espace des représentations, jamais dans l'espace des prix.
5. Entraînement auto-supervisé avec diagnostic d'effondrement.
6. Sonde de direction sur l'encodeur gelé, comparée à deux témoins (caractéristiques brutes, momentum).
7. Banc d'essai : simulation fidèle à `modules/backtest.js` d'Hermes, seuil choisi sur la validation, résultat lu sur le test seulement.
8. Export ONNX + statistiques de normalisation, prêts pour `onnxruntime-node`.

**Lancer** : menu *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Exécution → Tout exécuter*. Mode rapide : environ 15 à 20 minutes. Mode complet : 1 à 2 heures.

**À lire avant de croire un chiffre**

Un JEPA apprend une *représentation* du marché, pas une stratégie. L'avantage, s'il existe, se voit uniquement dans la sonde et dans le banc d'essai, sur des données que le modèle n'a jamais vues, et il doit battre le témoin sur caractéristiques brutes. Si ce n'est pas le cas, la représentation est jolie et inutile pour Hermes, et le carnet le dira. C'est le même principe que le chercheur de perles : pas de preuve sur données jamais vues, pas de trade.

In [ ]:
#@title 1. Réglages
RAPIDE = True  #@param {type:"boolean"}
# Les instruments : ceux qui reviennent dans l'univers d'Hermes (volume 24 h, actifs le week-end).
SYMBOLES = "BTCUSDT,ETHUSDT,SOLUSDT,XRPUSDT,DOGEUSDT,LTCUSDT,BCHUSDT,AVAXUSDT,LINKUSDT,FILUSDT,INJUSDT,ADAUSDT"  #@param {type:"string"}
DEBUT = "2023-01"  #@param {type:"string"}
FIN = ""  #@param {type:"string"}
INTERVALLE = "5m"
UTILISER_DRIVE = False  #@param {type:"boolean"}
GRAINE = 7

# Fenêtres, en bougies de 5 minutes.
CONTEXTE = 288        # 24 h vues par l'encodeur de contexte
CIBLE = 48            # 4 h à prédire dans l'espace des représentations
PATCH = 12            # 1 h par jeton
HORIZON = 12          # horizon de la sonde de direction : 1 h

# Les règles d'Hermes (modules/backtest.js, deploy/chercher_perles.js).
LEVIER = 15
FRAIS = 0.0005
SORTIES_HERMES = [
    dict(tp=0.80, act=0.30), dict(tp=0.60, act=0.20),
    dict(tp=0.40, act=0.30), dict(tp=0.30, act=0.15),
]
SL, CB = 0.30, 0.05
DUREES_H = [8, 12, 24]
MIN_WR_SEL, MIN_MOYENNE = 55.0, 0.02

# Modèle.
DIM, TETES, COUCHES, DIM_PRED, COUCHES_PRED = 128, 4, 4, 96, 2
MASQUE_CONTEXTE = 0.25
LOT = 256
LR = 3e-4
EMA_DEBUT, EMA_FIN = 0.996, 1.0

import os, sys, math, json, time, random, datetime as dt
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

if RAPIDE:
    SYMBOLES_LISTE = [s.strip() for s in SYMBOLES.split(",") if s.strip()][:6]
    EPOQUES, MAX_PAS_EPOQUE = 4, 1500
    auj = dt.date.today()
    DEBUT = (auj.replace(day=1) - dt.timedelta(days=365)).strftime("%Y-%m")
else:
    SYMBOLES_LISTE = [s.strip() for s in SYMBOLES.split(",") if s.strip()]
    EPOQUES, MAX_PAS_EPOQUE = 12, 10**9

if not FIN:
    auj = dt.date.today()
    FIN = (auj.replace(day=1) - dt.timedelta(days=1)).strftime("%Y-%m")   # dernier mois complet

random.seed(GRAINE); np.random.seed(GRAINE); torch.manual_seed(GRAINE)
APPAREIL = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RACINE = "/content"
if UTILISER_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RACINE = "/content/drive/MyDrive/hermes-jepa"
CACHE = os.path.join(RACINE, "cache-5m"); SORTIE = os.path.join(RACINE, "sortie")
os.makedirs(CACHE, exist_ok=True); os.makedirs(SORTIE, exist_ok=True)

print(f"mode {'RAPIDE' if RAPIDE else 'COMPLET'} | {len(SYMBOLES_LISTE)} instruments | {DEBUT} → {FIN} | {APPAREIL}")
if APPAREIL.type != "cuda":
    print("ATTENTION : pas de GPU. Exécution → Modifier le type d'exécution → GPU.")

## 2. Données

Source principale : `https://data.binance.vision/data/futures/um/monthly/klines/<SYMBOLE>/5m/<SYMBOLE>-5m-<AAAA-MM>.zip`. Ce sont les bougies officielles des perpétuels USDT-M, publiques, sans clé. Les fichiers récents portent une ligne d'en-tête et des horodatages en microsecondes : le lecteur gère les deux.

Repli : l'API publique OKX (`/api/v5/market/history-candles`), paginée, pour les instruments Hermes trade réellement. Elle remonte moins loin et ne donne pas la part acheteuse au marché, mise à zéro dans ce cas.

In [ ]:
#@title 2. Données — téléchargement et cache
import io, zipfile, requests

COLONNES = ["ts", "o", "h", "l", "c", "v", "ts_fin", "qv", "n", "tbv", "tbqv", "x"]

def mois_entre(debut, fin):
    a, m = map(int, debut.split("-")); fa, fm = map(int, fin.split("-"))
    out = []
    while (a, m) <= (fa, fm):
        out.append(f"{a:04d}-{m:02d}")
        m += 1
        if m == 13: a, m = a + 1, 1
    return out

def _get(url, essais=4):
    for k in range(essais):
        try:
            r = requests.get(url, timeout=60)
            if r.status_code == 404: return None
            r.raise_for_status()
            return r.content
        except Exception as e:
            if k == essais - 1: raise
            time.sleep(2 ** k)

def telecharger_mois_binance(symbole, mois):
    chemin = os.path.join(CACHE, f"{symbole}-{mois}.parquet")
    if os.path.exists(chemin): return pd.read_parquet(chemin)
    url = f"https://data.binance.vision/data/futures/um/monthly/klines/{symbole}/{INTERVALLE}/{symbole}-{INTERVALLE}-{mois}.zip"
    brut = _get(url)
    if brut is None: return None
    with zipfile.ZipFile(io.BytesIO(brut)) as z:
        nom = z.namelist()[0]
        with z.open(nom) as f:
            premiere = f.readline().decode("utf-8", "ignore")
        with z.open(nom) as f:
            df = pd.read_csv(f, header=0 if premiere.lower().startswith("open_time") else None, names=COLONNES)
    df = df[["ts", "o", "h", "l", "c", "v", "qv", "n", "tbv"]].copy()
    if df["ts"].iloc[0] > 1e14: df["ts"] = df["ts"] // 1000   # microsecondes depuis 2025
    df = df.astype({"ts": "int64", "o": "float64", "h": "float64", "l": "float64", "c": "float64",
                    "v": "float64", "qv": "float64", "n": "int64", "tbv": "float64"})
    df.to_parquet(chemin)
    return df

def telecharger_okx(symbole, n_max):
    inst = symbole.replace("USDT", "") + "-USDT-SWAP"
    lignes, apres = [], ""
    while len(lignes) < n_max:
        url = f"https://www.okx.com/api/v5/market/history-candles?instId={inst}&bar={INTERVALLE}&limit=100" + (f"&after={apres}" if apres else "")
        r = requests.get(url, timeout=30, headers={"User-Agent": "hermes-jepa"}); r.raise_for_status()
        d = r.json().get("data", [])
        if not d: break
        for k in d:
            if k[8] == "1":
                lignes.append([int(k[0]), float(k[1]), float(k[2]), float(k[3]), float(k[4]), float(k[5]), float(k[7]), 0, 0.0])
        apres = d[-1][0]
        time.sleep(0.12)
    df = pd.DataFrame(lignes, columns=["ts", "o", "h", "l", "c", "v", "qv", "n", "tbv"])
    return df.sort_values("ts").reset_index(drop=True)

def charger_symbole(symbole, mois_liste):
    morceaux, manquants = [], []
    for mois in mois_liste:
        try:
            df = telecharger_mois_binance(symbole, mois)
        except Exception as e:
            print(f"  {symbole} {mois} : {type(e).__name__} {e}"); df = None
        if df is None: manquants.append(mois)
        else: morceaux.append(df)
    source = "binance"
    if not morceaux:
        print(f"  {symbole} : Binance indisponible, repli OKX")
        df = telecharger_okx(symbole, n_max=len(mois_liste) * 8640); source = "okx"
    else:
        df = pd.concat(morceaux, ignore_index=True)
    df = df.drop_duplicates("ts").sort_values("ts").reset_index(drop=True)
    df = df[(df[["o", "h", "l", "c"]] > 0).all(axis=1)]
    pas = np.diff(df["ts"].values)
    trous = int((pas != 300000).sum())
    return df.reset_index(drop=True), source, manquants, trous

MOIS = mois_entre(DEBUT, FIN)
DONNEES = {}
lignes_resume = []
for s in SYMBOLES_LISTE:
    t0 = time.time()
    df, source, manquants, trous = charger_symbole(s, MOIS)
    if len(df) < CONTEXTE * 30:
        print(f"  {s} : trop court ({len(df)} bougies), écarté"); continue
    DONNEES[s] = df
    lignes_resume.append(dict(symbole=s, source=source, bougies=len(df),
        premiere=pd.to_datetime(df.ts.iloc[0], unit="ms").strftime("%Y-%m-%d"),
        derniere=pd.to_datetime(df.ts.iloc[-1], unit="ms").strftime("%Y-%m-%d"),
        mois_absents=len(manquants), trous=trous, s=round(time.time() - t0, 1)))
RESUME = pd.DataFrame(lignes_resume)
print(RESUME.to_string(index=False))
assert len(DONNEES) >= 2, "Il faut au moins deux instruments."

## 3. Caractéristiques

Onze valeurs par bougie, toutes **causales** (elles n'utilisent que le passé) et **sans dimension** (un log-rendement de BTC et de SHIB sont comparables, un prix ne l'est pas) :

| # | Nom | Définition |
|---|---|---|
| 0 | `r` | log(clôture / clôture précédente) |
| 1 | `hc` | log(haut / clôture) |
| 2 | `lc` | log(bas / clôture) |
| 3 | `oc` | log(ouverture / clôture) |
| 4 | `vol_z` | z-score causal de log(1 + volume) sur 288 bougies |
| 5 | `n_z` | z-score causal de log(1 + nombre de transactions) sur 288 bougies |
| 6 | `taker` | part acheteuse au marché moins 0,5 |
| 7 | `rv_z` | z-score causal de la volatilité réalisée sur 12 bougies |
| 8–9 | `tod_sin`, `tod_cos` | heure du jour |
| 10 | `dow_sin` | jour de la semaine |

Normalisation robuste (médiane, écart interquartile) calculée sur les bougies d'entraînement uniquement, puis bornée à ±8. Le fichier `normalisation.json` exporté contient ces constantes : Hermes devra recalculer les mêmes onze valeurs en JavaScript.

In [ ]:
#@title 3. Caractéristiques et découpage temporel
NOMS_CARACT = ["r", "hc", "lc", "oc", "vol_z", "n_z", "taker", "rv_z", "tod_sin", "tod_cos", "dow_sin"]
NB_CARACT = len(NOMS_CARACT)

def zscore_causal(x, fen=288):
    s = pd.Series(x)
    m = s.rolling(fen, min_periods=24).mean(); sd = s.rolling(fen, min_periods=24).std()
    return ((s - m) / (sd + 1e-9)).fillna(0.0).values

def caracteristiques(df):
    o, h, l, c, v = [df[k].values.astype(np.float64) for k in "ohlcv"]
    n = df["n"].values.astype(np.float64); tbv = df["tbv"].values.astype(np.float64)
    r = np.zeros_like(c); r[1:] = np.log(c[1:] / c[:-1])
    hc = np.log(h / c); lc = np.log(l / c); oc = np.log(o / c)
    vol_z = zscore_causal(np.log1p(v)); n_z = zscore_causal(np.log1p(n))
    with np.errstate(divide="ignore", invalid="ignore"):
        taker = np.where(v > 0, tbv / v, 0.5) - 0.5
    taker = np.clip(np.nan_to_num(taker), -0.5, 0.5)
    if not np.any(n): n_z = np.zeros_like(c)
    rv = pd.Series(r).rolling(12, min_periods=6).std().fillna(0.0).values
    rv_z = zscore_causal(np.log(rv + 1e-6))
    t = pd.to_datetime(df["ts"].values, unit="ms")
    tod = (t.hour * 60 + t.minute).values / 1440.0 * 2 * np.pi
    dow = t.dayofweek.values / 7.0 * 2 * np.pi
    X = np.stack([r, hc, lc, oc, vol_z, n_z, taker, rv_z, np.sin(tod), np.cos(tod), np.sin(dow)], axis=1)
    return X.astype(np.float32)

# Découpage par le temps, commun à tous les instruments : 70 % / 10 % / 20 %.
ts_min = min(int(df.ts.iloc[0]) for df in DONNEES.values())
ts_max = max(int(df.ts.iloc[-1]) for df in DONNEES.values())
T_TRAIN_FIN = ts_min + int(0.70 * (ts_max - ts_min))
T_VAL_FIN = ts_min + int(0.80 * (ts_max - ts_min))

BRUT = {s: caracteristiques(df) for s, df in DONNEES.items()}
train_bars = np.concatenate([X[DONNEES[s].ts.values < T_TRAIN_FIN] for s, X in BRUT.items()])
MEDIANE = np.median(train_bars, axis=0); IQR = np.subtract(*np.percentile(train_bars, [75, 25], axis=0)) + 1e-6
# Les caractéristiques d'horloge (sin/cos) n'ont pas besoin d'échelle.
MEDIANE[8:] = 0.0; IQR[8:] = 1.0
def normaliser(X): return np.clip((X - MEDIANE) / IQR, -8, 8).astype(np.float32)
CARACT = {s: normaliser(X) for s, X in BRUT.items()}
del train_bars

def indices_fenetres(df, futur):
    """Indices i (dernière bougie close vue) tels que [i-CONTEXTE+1, i+futur] existe et
       ne chevauche aucune frontière train/val/test."""
    ts = df.ts.values; n = len(ts)
    i = np.arange(CONTEXTE - 1, n - futur)
    debut, fin = ts[i - CONTEXTE + 1], ts[i + futur]
    tr = (fin < T_TRAIN_FIN)
    va = (debut >= T_TRAIN_FIN) & (fin < T_VAL_FIN)
    te = (debut >= T_VAL_FIN)
    return i[tr], i[va], i[te]

FENETRES = {s: indices_fenetres(df, CIBLE) for s, df in DONNEES.items()}
n_tr = sum(len(f[0]) for f in FENETRES.values()); n_va = sum(len(f[1]) for f in FENETRES.values()); n_te = sum(len(f[2]) for f in FENETRES.values())
print(f"fenêtres JEPA : train {n_tr:,} | val {n_va:,} | test {n_te:,}")
print("frontières :", pd.to_datetime(T_TRAIN_FIN, unit="ms"), "|", pd.to_datetime(T_VAL_FIN, unit="ms"))
print("médianes :", np.round(MEDIANE, 4)); print("IQR      :", np.round(IQR, 4))

## 4. Le modèle

Trois blocs, comme dans I-JEPA, transposés aux séries :

- **Encodeur de contexte** : les 288 bougies de contexte sont groupées en 24 jetons d'une heure, projetés, puis passés dans un Transformer. Pendant l'entraînement, un bloc contigu de jetons (25 %) est retiré au hasard pour que l'encodeur ne se repose pas sur la recopie du voisinage.
- **Encodeur cible** : même architecture, poids mis à jour par moyenne mobile exponentielle, jamais par gradient. Il voit les 28 jetons (contexte + 4 h futures) ; ses sorties sur les 4 jetons futurs, normalisées, sont la cible.
- **Prédicteur** : un petit Transformer qui reçoit les jetons de contexte et des jetons masqués portant la position des 4 heures futures, et doit produire les représentations cibles.

La perte est une L1 lissée entre prédictions et cibles. Aucun prix n'est jamais reconstruit : le modèle est libre d'ignorer le bruit de la microstructure et de garder ce qui est prévisible. Le diagnostic d'effondrement suit l'écart-type des représentations normalisées : proche de zéro, tout se ressemble et rien n'est appris.

In [ ]:
#@title 4. JEPA pour séries de bougies
import copy

N_CTX = CONTEXTE // PATCH; N_CIB = CIBLE // PATCH; N_TOT = N_CTX + N_CIB
assert CONTEXTE % PATCH == 0 and CIBLE % PATCH == 0

class Encodeur(nn.Module):
    def __init__(self, nb_caract, dim, tetes, couches, n_pos):
        super().__init__()
        self.patch = nn.Linear(PATCH * nb_caract, dim)
        self.pos = nn.Parameter(torch.zeros(1, n_pos, dim)); nn.init.trunc_normal_(self.pos, std=0.02)
        couche = nn.TransformerEncoderLayer(dim, tetes, dim * 4, dropout=0.1, activation="gelu", batch_first=True, norm_first=True)
        self.tr = nn.TransformerEncoder(couche, couches, enable_nested_tensor=False)
        self.norme = nn.LayerNorm(dim)
    def jetons(self, x):                      # x : (B, L, F) → (B, L/PATCH, dim)
        B, L, Fd = x.shape
        return self.patch(x.reshape(B, L // PATCH, PATCH * Fd))
    def forward(self, x, pos_idx=None):       # pos_idx : (B, n) indices de position des jetons conservés
        t = self.jetons(x)
        if pos_idx is None:
            t = t + self.pos[:, :t.shape[1]]
        else:
            t = torch.gather(t, 1, pos_idx.unsqueeze(-1).expand(-1, -1, t.shape[-1]))
            t = t + self.pos[0][pos_idx]
        return self.norme(self.tr(t))

class Predicteur(nn.Module):
    def __init__(self, dim, dim_pred, tetes, couches, n_pos):
        super().__init__()
        self.entree = nn.Linear(dim, dim_pred)
        self.pos = nn.Parameter(torch.zeros(1, n_pos, dim_pred)); nn.init.trunc_normal_(self.pos, std=0.02)
        self.masque = nn.Parameter(torch.zeros(1, 1, dim_pred)); nn.init.trunc_normal_(self.masque, std=0.02)
        couche = nn.TransformerEncoderLayer(dim_pred, tetes, dim_pred * 4, dropout=0.1, activation="gelu", batch_first=True, norm_first=True)
        self.tr = nn.TransformerEncoder(couche, couches, enable_nested_tensor=False)
        self.sortie = nn.Sequential(nn.LayerNorm(dim_pred), nn.Linear(dim_pred, dim))
    def forward(self, ctx, ctx_pos, cib_pos):
        B = ctx.shape[0]
        c = self.entree(ctx) + self.pos[0][ctx_pos]
        m = self.masque.expand(B, cib_pos.shape[1], -1) + self.pos[0][cib_pos]
        y = self.tr(torch.cat([c, m], dim=1))
        return self.sortie(y[:, ctx.shape[1]:])

class JEPA(nn.Module):
    def __init__(self):
        super().__init__()
        self.ctx = Encodeur(NB_CARACT, DIM, TETES, COUCHES, N_TOT)
        self.cible = copy.deepcopy(self.ctx)
        for p in self.cible.parameters(): p.requires_grad_(False)
        self.pred = Predicteur(DIM, DIM_PRED, TETES, COUCHES_PRED, N_TOT)
    @torch.no_grad()
    def ema(self, tau):
        for pc, pt in zip(self.ctx.parameters(), self.cible.parameters()):
            pt.mul_(tau).add_(pc.detach(), alpha=1 - tau)
    def forward(self, x, entrainement=True):   # x : (B, CONTEXTE+CIBLE, F)
        B = x.shape[0]; dev = x.device
        with torch.no_grad():
            t_all = self.cible(x)                            # (B, N_TOT, dim)
            cible = F.layer_norm(t_all[:, N_CTX:], (DIM,))   # cibles normalisées, comme I-JEPA
        cib_pos = torch.arange(N_CTX, N_TOT, device=dev).unsqueeze(0).expand(B, -1)
        if entrainement and MASQUE_CONTEXTE > 0:
            k = max(1, int(round(N_CTX * MASQUE_CONTEXTE)))
            debut = torch.randint(0, N_CTX - k + 1, (B, 1), device=dev)
            tous = torch.arange(N_CTX, device=dev).unsqueeze(0).expand(B, -1)
            garde = (tous < debut) | (tous >= debut + k)
            ctx_pos = tous[garde].view(B, N_CTX - k)
        else:
            ctx_pos = torch.arange(N_CTX, device=dev).unsqueeze(0).expand(B, -1)
        ctx = self.ctx(x[:, :CONTEXTE], ctx_pos)
        pred = self.pred(ctx, ctx_pos, cib_pos)
        perte = F.smooth_l1_loss(pred, cible)
        with torch.no_grad():
            z = F.normalize(ctx.mean(1), dim=-1)
            ecart = z.std(0).mean()                          # → 0 = effondrement
        return perte, ecart

modele = JEPA().to(APPAREIL)
print(f"paramètres : {sum(p.numel() for p in modele.parameters() if p.requires_grad):,} entraînables")

In [ ]:
#@title 5. Entraînement auto-supervisé
class Fenetres(torch.utils.data.Dataset):
    def __init__(self, part, futur):
        self.items = [(s, i) for s, f in FENETRES.items() for i in f[part]]
        self.futur = futur
    def __len__(self): return len(self.items)
    def __getitem__(self, k):
        s, i = self.items[k]
        return torch.from_numpy(CARACT[s][i - CONTEXTE + 1: i + 1 + self.futur])

ds_tr, ds_va = Fenetres(0, CIBLE), Fenetres(1, CIBLE)
dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=LOT, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
dl_va = torch.utils.data.DataLoader(ds_va, batch_size=LOT * 2, shuffle=False, num_workers=2)

pas_epoque = min(len(dl_tr), MAX_PAS_EPOQUE); pas_total = pas_epoque * EPOQUES; chauffe = min(500, pas_total // 10)
params = [p for p in modele.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.05, betas=(0.9, 0.95))
def lr_a(pas):
    if pas < chauffe: return LR * pas / max(1, chauffe)
    q = (pas - chauffe) / max(1, pas_total - chauffe); return LR * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * q)))
amp = APPAREIL.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp)

def evaluer():
    modele.eval(); pertes, ecarts = [], []
    with torch.no_grad():
        for x in dl_va:
            x = x.to(APPAREIL, non_blocking=True)
            with torch.autocast(device_type=APPAREIL.type, dtype=torch.float16, enabled=amp):
                p, e = modele(x, entrainement=False)
            pertes.append(p.item()); ecarts.append(e.item())
    modele.train(); return float(np.mean(pertes)), float(np.mean(ecarts))

JOURNAL = []; meilleur = float("inf"); pas = 0; t0 = time.time()
modele.train()
for ep in range(EPOQUES):
    it = iter(dl_tr)
    for _ in range(pas_epoque):
        x = next(it).to(APPAREIL, non_blocking=True)
        for g in opt.param_groups: g["lr"] = lr_a(pas)
        with torch.autocast(device_type=APPAREIL.type, dtype=torch.float16, enabled=amp):
            perte, ecart = modele(x)
        opt.zero_grad(set_to_none=True)
        scaler.scale(perte).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update()
        modele.ema(EMA_DEBUT + (EMA_FIN - EMA_DEBUT) * pas / max(1, pas_total))
        pas += 1
        if pas % 100 == 0:
            print(f"ép {ep+1}/{EPOQUES} pas {pas}/{pas_total} | perte {perte.item():.4f} | écart {ecart.item():.3f} | lr {lr_a(pas):.2e} | {time.time()-t0:.0f}s")
    pv, ev = evaluer()
    JOURNAL.append(dict(epoque=ep + 1, perte_val=pv, ecart_val=ev))
    print(f"== époque {ep+1} : perte val {pv:.4f} | écart val {ev:.3f}" + ("  (EFFONDREMENT probable)" if ev < 0.02 else ""))
    if pv < meilleur:
        meilleur = pv; torch.save(modele.state_dict(), os.path.join(SORTIE, "jepa.pt"))
modele.load_state_dict(torch.load(os.path.join(SORTIE, "jepa.pt"), map_location=APPAREIL))
print(f"terminé en {(time.time()-t0)/60:.1f} min ; meilleure perte val {meilleur:.4f}")

## 6. Sonde de direction

L'encodeur cible est gelé. Chaque fenêtre de contexte devient un vecteur (moyenne des jetons, concaténée au dernier jeton). Une régression logistique apprend à prédire le signe du rendement entre l'ouverture de la bougie suivante et la clôture une heure plus tard, exactement ce qu'Hermes encaisse. Les rendements plus petits que les frais aller-retour sont écartés de l'apprentissage.

Deux témoins reçoivent le même traitement : les 24 dernières bougies brutes aplaties, et le momentum (signe du rendement de la dernière heure). Si le JEPA ne bat pas le premier témoin sur le test, sa représentation n'apporte rien à Hermes.

In [ ]:
#@title 6. Sonde de direction et témoins
PAS_SONDE = 3 if RAPIDE else 2   # sous-échantillonnage des fenêtres d'apprentissage de la sonde (le test reste complet)
SEUIL_RENDEMENT = 2 * FRAIS       # sous les frais aller-retour, la direction n'a pas de valeur

@torch.no_grad()
def plonger(s, idx, lot=1024):
    """Représentation gelée des fenêtres de contexte finissant aux indices idx."""
    modele.eval(); X = CARACT[s]; out = []
    for k in range(0, len(idx), lot):
        b = idx[k:k + lot]
        x = torch.from_numpy(np.stack([X[i - CONTEXTE + 1: i + 1] for i in b])).to(APPAREIL)
        with torch.autocast(device_type=APPAREIL.type, dtype=torch.float16, enabled=amp):
            t = modele.cible(x)
        z = torch.cat([t.mean(1), t[:, -1]], dim=-1).float()
        out.append(z.cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, 2 * DIM), np.float32)

def brut_aplati(s, idx, n=24):
    X = CARACT[s]; return np.stack([X[i - n + 1: i + 1].reshape(-1) for i in idx]) if len(idx) else np.zeros((0, n * NB_CARACT), np.float32)

def rendement_futur(s, idx):
    df = DONNEES[s]; o = df.o.values; c = df.c.values
    return np.log(c[idx + HORIZON] / o[idx + 1])

def assembler(part, pas_ech):
    Z, R, S, I, B, M = [], [], [], [], [], []
    for s, f in FENETRES.items():
        idx = f[part][::pas_ech]
        idx = idx[idx + HORIZON < len(DONNEES[s])]
        if not len(idx): continue
        Z.append(plonger(s, idx)); B.append(brut_aplati(s, idx)); R.append(rendement_futur(s, idx))
        r1h = CARACT[s][:, 0]  # momentum : somme des 12 derniers log-rendements normalisés
        M.append(np.array([r1h[i - 11: i + 1].sum() for i in idx]))
        S += [s] * len(idx); I.append(idx)
    return dict(Z=np.concatenate(Z), B=np.concatenate(B), r=np.concatenate(R), mom=np.concatenate(M), sym=np.array(S), idx=np.concatenate(I))

t0 = time.time()
E_TR, E_VA, E_TE = assembler(0, PAS_SONDE), assembler(1, 1), assembler(2, 1)
print(f"plongements : train {len(E_TR['r']):,} | val {len(E_VA['r']):,} | test {len(E_TE['r']):,} | {time.time()-t0:.0f}s")

def sonde_logistique(Xtr, ytr, Xva, yva, wds=(1e-4, 1e-3, 1e-2, 1e-1), pas=400):
    """Régression logistique sur GPU ; le poids de régularisation est choisi sur la validation."""
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    def T(a): return torch.from_numpy(((a - mu) / sd).astype(np.float32)).to(APPAREIL)
    xt, yt = T(Xtr), torch.from_numpy(ytr.astype(np.float32)).to(APPAREIL)
    xv, yv = T(Xva), torch.from_numpy(yva.astype(np.float32)).to(APPAREIL)
    meilleur = None
    for wd in wds:
        lin = nn.Linear(xt.shape[1], 1).to(APPAREIL); o = torch.optim.Adam(lin.parameters(), lr=1e-2, weight_decay=wd)
        for _ in range(pas):
            o.zero_grad(); F.binary_cross_entropy_with_logits(lin(xt).squeeze(-1), yt).backward(); o.step()
        with torch.no_grad(): pv = F.binary_cross_entropy_with_logits(lin(xv).squeeze(-1), yv).item()
        if meilleur is None or pv < meilleur[0]: meilleur = (pv, wd, copy.deepcopy(lin))
    pv, wd, lin = meilleur
    def proba(X):
        with torch.no_grad(): return torch.sigmoid(lin(T(X)).squeeze(-1)).cpu().numpy()
    return dict(lin=lin, mu=mu, sd=sd, wd=wd, perte_val=pv, proba=proba)

def etiquettes(E):
    garde = np.abs(E["r"]) > SEUIL_RENDEMENT
    return garde, (E["r"] > 0).astype(np.float32)

g_tr, y_tr = etiquettes(E_TR); g_va, y_va = etiquettes(E_VA); g_te, y_te = etiquettes(E_TE)
SONDES = {
    "JEPA": sonde_logistique(E_TR["Z"][g_tr], y_tr[g_tr], E_VA["Z"][g_va], y_va[g_va]),
    "brut 24 bougies": sonde_logistique(E_TR["B"][g_tr], y_tr[g_tr], E_VA["B"][g_va], y_va[g_va]),
}
def auc(p, y):
    ordre = np.argsort(p); rang = np.empty_like(ordre, dtype=np.float64); rang[ordre] = np.arange(1, len(p) + 1)
    n1 = y.sum(); n0 = len(y) - n1
    return float((rang[y == 1].sum() - n1 * (n1 + 1) / 2) / max(1, n1 * n0))

lignes = []
for nom, sd in SONDES.items():
    p = sd["proba"](E_TE["Z"] if nom == "JEPA" else E_TE["B"])
    y = y_te
    ligne = dict(sonde=nom, wd=sd["wd"], auc_test=round(auc(p[g_te], y[g_te]), 4),
                 exactitude_test=round(float(((p[g_te] > 0.5) == (y[g_te] > 0.5)).mean() * 100), 2))
    for seuil in (0.55, 0.60, 0.65):
        conf = np.maximum(p, 1 - p) >= seuil; couv = conf.mean()
        sens = np.where(p >= 0.5, 1, -1); juste = (sens == np.where(E_TE["r"] > 0, 1, -1))[conf & g_te]
        ligne[f"préc@{seuil}"] = round(float(juste.mean() * 100), 1) if len(juste) else float("nan")
        ligne[f"couv@{seuil}"] = round(float(couv * 100), 1)
    lignes.append(ligne)
# Témoin momentum : le signe de la dernière heure, sans apprentissage.
sens_m = np.where(E_TE["mom"] > 0, 1, -1); juste_m = (sens_m == np.where(E_TE["r"] > 0, 1, -1))[g_te]
lignes.append(dict(sonde="momentum 1 h", exactitude_test=round(float(juste_m.mean() * 100), 2)))
TABLE_SONDE = pd.DataFrame(lignes); print(TABLE_SONDE.to_string(index=False))
print("\nLecture : préc@0.60 est la précision directionnelle des seules fenêtres où la sonde est sûre à 60 % ou plus ;")
print("couv@0.60 la part des fenêtres concernées. Le JEPA doit dépasser le témoin brut, sinon il n'apporte rien.")

## 7. Banc d'essai aux règles d'Hermes

Réécriture fidèle de `modules/backtest.js` : entrée à l'ouverture de la bougie suivante, TP / SL en pourcentage de la marge convertis en prix par le levier, trail armé à `act` et rappelé de `cb`, fermeture à l'échéance, frais taker sur les deux jambes, toute ambiguïté intra-bougie tranchée contre la stratégie, une position à la fois par instrument. Les mêmes quatre sorties et trois durées que le chercheur de perles sont essayées.

**Protocole** : la combinaison (seuil de confiance, sortie, durée) est choisie sur la **validation**. Le **test** n'est lu qu'une fois, avec cette seule combinaison. C'est la règle du chercheur de perles, sans repêchage.

In [ ]:
#@title 7. Banc d'essai (simulation fidèle à modules/backtest.js)
def simuler(o, h, l, c, ts, signaux, sortie, lev=LEVIER, frais=FRAIS):
    tp = sortie["tp"] / lev; sl = sortie["sl"] / lev; act = sortie["act"] / lev; cb = sortie["cb"] / lev
    hold = sortie["hold_h"] * 3600000; frais_marge = 2 * frais * lev
    trades = []; pos = None
    for i in range(len(c)):
        if pos is None and i > 0 and signaux[i - 1] != 0:
            pos = dict(i=i, ts=ts[i], px=o[i], dir=int(signaux[i - 1]), pic=None)
        if pos is None: continue
        long = pos["dir"] > 0; px = pos["px"]
        px_tp = px * (1 + (tp if long else -tp)); px_sl = px * (1 - (sl if long else -sl)); px_act = px * (1 + (act if long else -act))
        sortie_px = None; raison = None
        stop = px_sl
        if pos["pic"] is not None:
            px_trail = pos["pic"] * (1 - (cb if long else -cb))
            stop = max(stop, px_trail) if long else min(stop, px_trail)
        if (o[i] <= stop) if long else (o[i] >= stop): sortie_px, raison = o[i], "gap-stop"
        elif (l[i] <= stop) if long else (h[i] >= stop): sortie_px, raison = stop, ("trail" if pos["pic"] is not None and stop != px_sl else "sl")
        elif (h[i] >= px_tp) if long else (l[i] <= px_tp): sortie_px, raison = px_tp, "tp"
        if sortie_px is None:
            ext = h[i] if long else l[i]
            if (ext >= px_act) if long else (ext <= px_act):
                pos["pic"] = ext if pos["pic"] is None else (max(pos["pic"], ext) if long else min(pos["pic"], ext))
            if ts[i] + 300000 >= pos["ts"] + hold: sortie_px, raison = c[i], "hold"
        if sortie_px is not None:
            brut = pos["dir"] * (sortie_px - px) / px * lev
            trades.append(dict(ts_in=int(pos["ts"]), dir=pos["dir"], raison=raison, pnl=brut - frais_marge))
            pos = None
    return trades

def resumer(trades):
    n = len(trades); g = sum(1 for t in trades if t["pnl"] > 0); net = sum(t["pnl"] for t in trades)
    def sens(d):
        l = [t for t in trades if t["dir"] == d]; gg = sum(1 for t in l if t["pnl"] > 0)
        return dict(trades=len(l), winrate=100 * gg / len(l) if l else 0.0)
    return dict(trades=n, winrate=100 * g / n if n else 0.0, net=net, moyenne=net / n if n else 0.0, longs=sens(1), shorts=sens(-1))

def serie_signaux(E, sonde_nom, seuil):
    """Pour chaque instrument, un tableau aligné sur ses bougies : 1, -1 ou 0."""
    p = SONDES[sonde_nom]["proba"](E["Z"] if sonde_nom == "JEPA" else E["B"])
    sens = np.where(p >= 0.5, 1, -1) * (np.maximum(p, 1 - p) >= seuil)
    out = {}
    for s in DONNEES:
        m = E["sym"] == s; sig = np.zeros(len(DONNEES[s]), np.int8); sig[E["idx"][m]] = sens[m]; out[s] = sig
    return out

def banc(E, sonde_nom, seuil, sortie):
    signaux = serie_signaux(E, sonde_nom, seuil); tous = []; par_sym = {}
    for s, df in DONNEES.items():
        o, h, l, c, ts = df.o.values, df.h.values, df.l.values, df.c.values, df.ts.values
        tr = simuler(o, h, l, c, ts, signaux[s], sortie); par_sym[s] = resumer(tr); tous += tr
    return resumer(tous), par_sym, tous

# Choix sur la validation : le winrate tranche, le gain net départage, sous les mêmes portes que le chercheur.
CHOIX = {}
for nom in SONDES:
    candidats = []
    for seuil in (0.55, 0.60, 0.65, 0.70):
        for sb in SORTIES_HERMES:
            for hh in DUREES_H:
                sortie = dict(tp=sb["tp"], act=sb["act"], sl=SL, cb=CB, hold_h=hh)
                r, _, _ = banc(E_VA, nom, seuil, sortie)
                if r["trades"] >= 12: candidats.append((r["winrate"], r["net"], seuil, sortie, r))
    if not candidats: print(f"{nom} : aucune combinaison ne donne 12 trades en validation"); continue
    # Les portes du chercheur d'abord (gain net positif, winrate, gain moyen) ; le classement ensuite.
    passent = [c for c in candidats if c[1] > 0 and c[0] >= MIN_WR_SEL and c[4]["moyenne"] >= MIN_MOYENNE]
    porte_val = bool(passent)
    (passent or candidats).sort(key=lambda x: (x[0], x[1]), reverse=True)
    wr, net, seuil, sortie, r = (passent or candidats)[0]; CHOIX[nom] = (seuil, sortie)
    print(f"{nom:>16} | validation : seuil {seuil} tp {sortie['tp']} act {sortie['act']} hold {sortie['hold_h']}h | {r['trades']} trades wr {wr:.0f}% net {net:.2f} moyenne {r['moyenne']:.3f}"
          + ("" if porte_val else "  (AUCUNE combinaison ne passe les portes en validation : le test ci-dessous est donné pour information)"))

print("\n=== TEST (lu une seule fois, avec la combinaison choisie sur la validation) ===")
RESULTATS = {}
for nom, (seuil, sortie) in CHOIX.items():
    r, par_sym, trades = banc(E_TE, nom, seuil, sortie); RESULTATS[nom] = dict(seuil=seuil, sortie=sortie, total=r, par_symbole=par_sym)
    jours = (ts_max - T_VAL_FIN) / 86400000
    porte = "PASSE les portes du chercheur" if (r["winrate"] >= MIN_WR_SEL and r["moyenne"] >= MIN_MOYENNE and r["trades"] >= 12) else "ne passe PAS les portes du chercheur"
    print(f"\n{nom} : {r['trades']} trades ({r['trades']/max(1,jours):.1f}/jour) | winrate {r['winrate']:.1f}% | net {r['net']:.2f} | moyenne {r['moyenne']:.3f} de marge/trade | "
          f"longs {r['longs']['trades']} à {r['longs']['winrate']:.0f}% · shorts {r['shorts']['trades']} à {r['shorts']['winrate']:.0f}% → {porte}")
    print(pd.DataFrame([dict(symbole=s, trades=v["trades"], winrate=round(v["winrate"], 1), net=round(v["net"], 2), moyenne=round(v["moyenne"], 3)) for s, v in par_sym.items()]).to_string(index=False))

## 8. Export pour Hermes

Le fichier `hermes_jepa.onnx` prend une fenêtre `(1, 288, 11)` de caractéristiques normalisées et renvoie `(1, 2)` : probabilité de baisse puis de hausse. `normalisation.json` porte les médianes, les écarts interquartiles, l'ordre des caractéristiques, le seuil et la sortie retenus, pour recalculer exactement la même entrée en JavaScript.

Brancher dans Hermes revient à ajouter un quatorzième signal `jepa` dans `modules/signaux.js` (chargé par `onnxruntime-node`, évalué sur les 299 bougies closes comme les autres), puis à laisser le **chercheur de perles le juger comme n'importe quel signal**. Rien ne court-circuite la règle « pas de perle = pas de trade ».

In [ ]:
#@title 8. Export ONNX et rapport
import shutil
class ModeleHermes(nn.Module):
    def __init__(self, enc, sonde):
        super().__init__()
        self.enc = copy.deepcopy(enc).float().eval()
        self.mu = nn.Parameter(torch.from_numpy(sonde["mu"].astype(np.float32)), requires_grad=False)
        self.sd = nn.Parameter(torch.from_numpy(sonde["sd"].astype(np.float32)), requires_grad=False)
        self.lin = copy.deepcopy(sonde["lin"]).float().cpu()
    def forward(self, x):
        t = self.enc(x); z = torch.cat([t.mean(1), t[:, -1]], dim=-1)
        p_hausse = torch.sigmoid(self.lin((z - self.mu) / self.sd))
        return torch.cat([1 - p_hausse, p_hausse], dim=-1)

if "JEPA" in SONDES:
    mh = ModeleHermes(modele.cible.cpu(), SONDES["JEPA"]).eval()
    modele.cible.to(APPAREIL)
    exemple = torch.zeros(1, CONTEXTE, NB_CARACT)
    chemin_onnx = os.path.join(SORTIE, "hermes_jepa.onnx")
    # Le chemin rapide fusionné du Transformer (aten::_transformer_encoder_layer_fwd) n'a pas d'équivalent ONNX.
    try: torch.backends.mha.set_fastpath_enabled(False)
    except Exception: pass
    torch.onnx.export(mh, exemple, chemin_onnx, input_names=["fenetre"], output_names=["proba"],
                      dynamic_axes={"fenetre": {0: "lot"}, "proba": {0: "lot"}}, opset_version=17, dynamo=False)
    # Vérification : ONNX Runtime doit rendre les mêmes probabilités que PyTorch.
    try:
        import onnxruntime as ort
    except ImportError:
        os.system("pip install -q onnxruntime"); import onnxruntime as ort
    s0 = next(iter(DONNEES)); i0 = FENETRES[s0][2][0] if len(FENETRES[s0][2]) else FENETRES[s0][0][-1]
    xv = torch.from_numpy(CARACT[s0][i0 - CONTEXTE + 1: i0 + 1][None])
    with torch.no_grad(): ref = mh(xv).numpy()
    sess = ort.InferenceSession(chemin_onnx, providers=["CPUExecutionProvider"])
    out = sess.run(None, {"fenetre": xv.numpy()})[0]
    print("PyTorch", np.round(ref, 4), "| ONNX", np.round(out, 4), "| écart max", float(np.abs(ref - out).max()))

    choix = CHOIX.get("JEPA", (None, None))
    json.dump(dict(caracteristiques=NOMS_CARACT, mediane=MEDIANE.tolist(), iqr=IQR.tolist(), borne=8,
                   contexte=CONTEXTE, patch=PATCH, horizon=HORIZON, intervalle=INTERVALLE,
                   seuil=choix[0], sortie=choix[1], levier=LEVIER, frais=FRAIS,
                   sonde=dict(mu=mh.mu.tolist(), sd=mh.sd.tolist())),
              open(os.path.join(SORTIE, "normalisation.json"), "w"), indent=1)
json.dump(dict(mode="rapide" if RAPIDE else "complet", instruments=SYMBOLES_LISTE, periode=[DEBUT, FIN],
               frontieres=[int(T_TRAIN_FIN), int(T_VAL_FIN)], donnees=RESUME.to_dict("records"),
               entrainement=JOURNAL, sonde=TABLE_SONDE.to_dict("records"), banc=RESULTATS),
          open(os.path.join(SORTIE, "rapport.json"), "w"), indent=1, default=float)
archive = shutil.make_archive(os.path.join(RACINE, "hermes-jepa-sortie"), "zip", SORTIE)
print("archive :", archive)
try:
    from google.colab import files; files.download(archive)
except Exception as e:
    print("téléchargement manuel :", archive)

## 9. Lire les résultats

**Ce qui compterait comme une vraie trouvaille**, sur le test, jamais sur la validation :

- la sonde JEPA a une AUC nettement supérieure à 0,5 **et** supérieure au témoin brut ;
- à 60 % de confiance, sa précision dépasse 58 % avec une couverture d'au moins quelques pour cent des fenêtres ;
- le banc d'essai passe les portes du chercheur (winrate ≥ 55 %, gain moyen ≥ 0,02 de marge par trade, au moins 12 trades), sur plusieurs instruments et non un seul.

**Ce que signifie un résultat plat.** Une AUC de 0,50 à 0,52 sur une heure de bougies de 5 minutes est le cas général en crypto liquide : l'information publique est arbitrée en quelques secondes, et le JEPA apprend surtout le régime de volatilité, pas la direction. Ce n'est pas un échec du carnet, c'est une mesure. Les leviers qui changent réellement l'issue, par ordre de rendement attendu : un horizon plus long (4 h ou 24 h, avec des sorties adaptées), des caractéristiques que le marché n'a pas déjà dans le prix (financement, intérêt ouvert, liquidations, carnet), et davantage d'instruments moins liquides, là où Hermes trouve déjà ses perles.

**Ce qui ne doit pas changer**, quel que soit le résultat : le chercheur de perles reste le juge, et le moteur n'ouvre que sur une perle validée sur des jours jamais regardés.